# 21 - Introduccion a ciencia de datos: CRISP-DM, prediccion e Iris

## Objetivos de aprendizaje

En esta sesion aprenderas a:

1. Entender el marco CRISP-DM como ruta de un proyecto de datos.
2. Definir con claridad que significa predecir.
3. Construir un predictor simple con reglas `if/else`.
4. Ver por que muchos algoritmos de prediccion dividen el espacio en regiones.
5. Comprender de forma intuitiva la funcion de densidad de los datos.
6. Practicar con ejercicios que obligan a razonar, no solo copiar codigo.


## Ruta de la sesion

1. CRISP-DM: de la pregunta de negocio al despliegue.
2. Que es predecir y que no es predecir.
3. Iris: exploracion rapida.
4. Prediccion con `if/else`.
5. Regiones de decision (regla manual, arbol y k-NN).
6. Densidad de datos: histograma y una KDE simple.
7. Ejercicios de pensamiento.


## 1) CRISP-DM en lenguaje practico

CRISP-DM (Cross-Industry Standard Process for Data Mining) es un modelo de trabajo para proyectos de datos.
No es un algoritmo: es una guia de proceso.

Sus 6 fases:

1. **Entendimiento del negocio**: cual es la decision que quieres mejorar.
2. **Entendimiento de los datos**: que datos hay, calidad, sesgos, faltantes.
3. **Preparacion de datos**: limpieza, transformaciones, seleccion de variables.
4. **Modelado**: entrenar modelos (desde reglas simples hasta modelos complejos).
5. **Evaluacion**: medir si el modelo sirve para la meta real, no solo metricas bonitas.
6. **Despliegue**: poner el modelo en uso y monitorear si sigue funcionando.

Importante: CRISP-DM no es lineal. Se itera constantemente.


## 2) Que significa predecir

Predecir es usar patrones observados en datos pasados para estimar un valor o categoria no vista.

- Si la salida es una **categoria** (ej. especie de flor), es clasificacion.
- Si la salida es un **numero** (ej. precio), es regresion.

Predecir no es magia:

- no garantiza certeza,
- depende de la calidad de datos,
- siempre tiene error e incertidumbre.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.metrics import confusion_matrix
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier


In [ ]:
# Cargamos Iris desde scikit-learn.
iris = load_iris()
X = iris.data
y = iris.target
target_names = iris.target_names
feature_names = iris.feature_names

print("Forma de X:", X.shape)
print("Forma de y:", y.shape)
print("Clases:", target_names)
print("Variables:", feature_names)


## 3) Exploracion rapida de Iris

Nos enfocaremos en dos variables faciles de visualizar:

- `petal length (cm)`
- `petal width (cm)`

Con dos variables ya podemos ver separaciones claras entre especies.


In [ ]:
X2 = X[:, [2, 3]]  # petal length, petal width

colors = ["tab:blue", "tab:orange", "tab:green"]

plt.figure(figsize=(8, 5))
for k, color in enumerate(colors):
    mask = y == k
    plt.scatter(
        X2[mask, 0],
        X2[mask, 1],
        alpha=0.75,
        color=color,
        label=target_names[k],
    )

plt.xlabel("petal length (cm)")
plt.ylabel("petal width (cm)")
plt.title("Iris en dos variables")
plt.legend()
plt.grid(alpha=0.2)
plt.show()


## 4) Un predictor simple con `if/else`

Empezamos con un modelo interpretable y manual.

Regla:

1. Si `petal length < 2.45` -> `setosa`.
2. En otro caso, si `petal width < 1.75` -> `versicolor`.
3. En otro caso -> `virginica`.

Esto ya es un clasificador real, aunque simple.


In [ ]:
def predict_if_else(row):
    petal_length = row[0]
    petal_width = row[1]

    if petal_length < 2.45:
        return 0  # setosa
    elif petal_width < 1.75:
        return 1  # versicolor
    else:
        return 2  # virginica


print("Ejemplo [1.4, 0.2] ->", target_names[predict_if_else(np.array([1.4, 0.2]))])
print("Ejemplo [5.2, 2.0] ->", target_names[predict_if_else(np.array([5.2, 2.0]))])


In [ ]:
y_pred_rule = np.array([predict_if_else(row) for row in X2])
acc_rule = (y_pred_rule == y).mean()
cm_rule = confusion_matrix(y, y_pred_rule)

print(f"Accuracy de la regla if/else: {acc_rule:.3f}")
print("\nMatriz de confusion (filas = real, columnas = predicho):")
print(cm_rule)


In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm_rule, cmap="Blues")

ax.set_xticks(np.arange(len(target_names)))
ax.set_yticks(np.arange(len(target_names)))
ax.set_xticklabels(target_names)
ax.set_yticklabels(target_names)
ax.set_xlabel("Predicho")
ax.set_ylabel("Real")
ax.set_title("Matriz de confusion - Regla if/else")

for i in range(cm_rule.shape[0]):
    for j in range(cm_rule.shape[1]):
        ax.text(j, i, cm_rule[i, j], ha="center", va="center", color="black")

fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()


## 5) Muchos algoritmos predicen dividiendo en regiones

Intuicion:

- El plano de variables se divide en zonas.
- Cada zona queda etiquetada con una clase.
- Predecir es preguntar: "en que zona cayo este punto?"

Vamos a comparar 3 formas:

1. Regla manual `if/else` (fronteras rectas definidas por nosotros).
2. Arbol de decision (fronteras rectas aprendidas).
3. k-NN (fronteras mas flexibles por vecindad).


In [ ]:
tree = DecisionTreeClassifier(max_depth=3, random_state=42)
tree.fit(X2, y)

knn = KNeighborsClassifier(n_neighbors=11)
knn.fit(X2, y)


def plot_regions(ax, predict_fn, title):
    x_min, x_max = X2[:, 0].min() - 0.5, X2[:, 0].max() + 0.5
    y_min, y_max = X2[:, 1].min() - 0.5, X2[:, 1].max() + 0.5
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 300),
        np.linspace(y_min, y_max, 300),
    )

    grid = np.c_[xx.ravel(), yy.ravel()]
    zz = predict_fn(grid).reshape(xx.shape)

    ax.contourf(xx, yy, zz, alpha=0.25, levels=[-0.5, 0.5, 1.5, 2.5], cmap="viridis")

    for k, color in enumerate(colors):
        mask = y == k
        ax.scatter(X2[mask, 0], X2[mask, 1], s=18, color=color, label=target_names[k], alpha=0.85)

    ax.set_title(title)
    ax.set_xlabel("petal length (cm)")
    ax.set_ylabel("petal width (cm)")
    ax.grid(alpha=0.15)


fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharex=True, sharey=True)

plot_regions(
    axes[0],
    lambda grid: np.array([predict_if_else(row) for row in grid]),
    "Regla manual if/else",
)
plot_regions(axes[1], lambda grid: tree.predict(grid), "Arbol de decision")
plot_regions(axes[2], lambda grid: knn.predict(grid), "k-NN")

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=3)
plt.tight_layout(rect=[0, 0.08, 1, 1])
plt.show()


Observa:

- La regla manual usa pocas fronteras y es facil de explicar.
- El arbol aprende cortes alineados con ejes.
- k-NN puede hacer fronteras mas "organicas" segun vecinos.

La idea comun es la misma: **particionar el espacio** en regiones de prediccion.


## 6) Funcion de densidad de forma intuitiva

Piensa en la densidad como "donde se acumulan mas observaciones".

- Si la curva es alta en una zona, hay muchos datos por ahi.
- Si la curva es baja, esa zona es rara.

Primero con histograma (aproximacion discreta), luego con una KDE sencilla.


In [ ]:
petal_length = X2[:, 0]

plt.figure(figsize=(8, 5))
bins = np.linspace(petal_length.min() - 0.2, petal_length.max() + 0.2, 18)

for k, color in enumerate(colors):
    mask = y == k
    plt.hist(
        petal_length[mask],
        bins=bins,
        density=True,
        alpha=0.35,
        color=color,
        label=f"{target_names[k]} (densidad aprox)",
    )

plt.xlabel("petal length (cm)")
plt.ylabel("densidad aproximada")
plt.title("Histograma normalizado por especie")
plt.legend()
plt.grid(alpha=0.2)
plt.show()


In [ ]:
def gaussian_kde_1d(samples, grid, bandwidth=0.18):
    """KDE 1D minima para intuicion: promedio de kernels gaussianos."""
    samples = np.asarray(samples)
    grid = np.asarray(grid)
    z = (grid[None, :] - samples[:, None]) / bandwidth
    kernels = np.exp(-0.5 * z ** 2) / np.sqrt(2 * np.pi)
    return kernels.mean(axis=0) / bandwidth


grid = np.linspace(petal_length.min() - 0.4, petal_length.max() + 0.4, 350)

plt.figure(figsize=(8, 5))
for k, color in enumerate(colors):
    mask = y == k
    density = gaussian_kde_1d(petal_length[mask], grid, bandwidth=0.20)
    area = np.trapz(density, grid)
    plt.plot(grid, density, color=color, lw=2, label=f"{target_names[k]} (area~{area:.2f})")

plt.xlabel("petal length (cm)")
plt.ylabel("densidad")
plt.title("KDE simple por especie (interpretacion intuitiva)")
plt.legend()
plt.grid(alpha=0.2)
plt.show()


Lectura rapida de la densidad:

1. Si dos curvas se superponen mucho, la clasificacion en esa zona sera mas dificil.
2. Si una especie concentra masa en una region separada, predecir sera mas facil.
3. La densidad no dice causalidad: solo describe como se distribuyen los datos.


## 7) Ejercicios de pensamiento (no copiar/pegar)

Escribe primero tu razonamiento en texto y luego programa.


### Ejercicio 1: CRISP-DM aplicado a un caso real

Elige uno:

- desercion estudiantil,
- prediccion de demanda de biblioteca,
- deteccion de retraso de pagos.

Para cada fase de CRISP-DM responde:

1. Que decision quieres mejorar.
2. Que dato minimo necesitas y que riesgo de sesgo hay.
3. Como evaluarias si el modelo sirve en la practica.


### Ejercicio 2: antes de correr codigo, predice a mano

Sin ejecutar nada, decide que clase daria la regla `if/else` para estos puntos:

- A = [1.9, 0.3]
- B = [4.7, 1.4]
- C = [5.9, 2.1]

Luego ejecuta y compara: donde acertaste y por que.


In [ ]:
# Celda para comprobar el Ejercicio 2.
puntos = np.array([
    [1.9, 0.3],
    [4.7, 1.4],
    [5.9, 2.1],
])

for i, p in enumerate(puntos, start=1):
    pred = predict_if_else(p)
    print(f"Punto {i} {p} -> {target_names[pred]}")


### Ejercicio 3: diseña una mejor regla

Sin usar modelos de scikit-learn, ajusta los umbrales de `if/else` para subir accuracy.

Condiciones:

1. Solo puedes usar `petal length` y `petal width`.
2. Maximo 3 condiciones `if/elif`.
3. Debes explicar por que elegiste esos cortes.


In [ ]:
# Escribe aqui tu nueva regla y compara accuracy con la original.
# Pista: observa donde se concentran errores en la matriz de confusion.


### Ejercicio 4: regiones y complejidad

Compara visualmente las regiones de `if/else`, arbol y k-NN.
Luego responde:

1. Cual modelo parece mas interpretable para explicar a una persona no tecnica.
2. Cual parece mas flexible.
3. Donde ves riesgo de sobreajuste.


### Ejercicio 5: densidad e incertidumbre

Usa la grafica de densidad y responde:

1. En que rangos de `petal length` esperas mas confusiones entre clases.
2. Si recibes una nueva flor con `petal length = 4.9`, que clase te parece mas probable y con que nivel de confianza (alto/medio/bajo).
3. Que informacion adicional te gustaria medir para reducir incertidumbre.


In [ ]:
# Espacio para notas y comprobaciones del Ejercicio 5.


## 8) Cierre

Ideas clave:

1. CRISP-DM organiza el trabajo tecnico alrededor de una decision real.
2. Predecir es estimar bajo incertidumbre, no adivinar con certeza.
3. Una regla `if/else` ya puede funcionar como clasificador interpretable.
4. Muchos algoritmos de clasificacion particionan el espacio en regiones.
5. La densidad ayuda a entender donde hay evidencia fuerte y donde hay ambiguedad.
